# Supplementary: Entropy Estimation Methods

This notebook compares entropy estimation methods for the human response distributions used in the analysis pipeline.

**Goals:**
1. Verify that the current pipeline's Shannon entropy (via `scipy.stats.entropy` on MLE probabilities) matches `infomeasure`'s `discrete` estimator
2. Explore small-sample bias-corrected estimators appropriate for N≈50 responses per distribution

**Key context:**
- ~50 human participants contribute one response (word) per word_index
- Current pipeline: `scipy.stats.entropy(counts/N)` — MLE plug-in estimator, known to underestimate entropy for small N
- Candidate corrected methods: Miller-Madow, Shrinkage (James-Stein), NSB, Chao-Shen

This notebook uses a small subset of data (1 task × 1 modality × 20 word indices) for fast iteration.

## Setup

Requires: `pip install infomeasure`

In [ ]:
import os, sys
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

sys.path.append('/dartfs/rc/lab/F/FinnLab/tommy/isc_asynchrony_behavior/code/utils/')
from config import BASE_DIR

try:
    import infomeasure as im
    print(f'infomeasure imported successfully')
except ImportError:
    raise ImportError('Please install infomeasure: pip install infomeasure')

BEHAVIORAL_DIR = os.path.join(BASE_DIR, 'osf', 'results', 'behavioral')

## 1. Load data

We load a small slice of the subject-level behavioral data (responses from individual participants) and the pre-computed group-level analysis file (which contains the current pipeline's entropy values).

In [ ]:
# --- Configuration ---
TASK = 'black'
MODALITY = 'audio'
N_WORDS = 20   # number of word indices to examine

# Load subject-level responses (one row per participant × word_index)
df_subjects = pd.read_csv(os.path.join(BEHAVIORAL_DIR, 'all-task_subject-behavior_lemmatized.csv'))
print(f'Subject file shape: {df_subjects.shape}')
print(f'Columns: {list(df_subjects.columns)}')

# Load group-level analyzed behavior (one row per word_index, contains current entropy values)
df_analyzed = pd.read_csv(os.path.join(BEHAVIORAL_DIR, 'all-task_group-analyzed-behavior_human-lemmatized.csv'))
print(f'\nAnalyzed file shape: {df_analyzed.shape}')
print(f'Columns: {list(df_analyzed.columns)}')

In [ ]:
# Subset to our small sample
df_sub = df_subjects[(df_subjects['task'] == TASK) & (df_subjects['modality'] == MODALITY)].copy()
sample_words = sorted(df_sub['word_index'].unique())[:N_WORDS]
df_sub = df_sub[df_sub['word_index'].isin(sample_words)]

print(f'Subset: task={TASK}, modality={MODALITY}')
print(f'Word indices: {sample_words}')
print(f'Responses per word index:')
print(df_sub.groupby('word_index')['response'].count().to_string())

## 2. Replicate the current pipeline's entropy computation

The current `analyze_human_results` function in `analysis_utils.py` computes:
```python
unique, counts = np.unique(responses, return_counts=True)
probs = counts / sum(counts)           # MLE plug-in probabilities
entropy = scipy.stats.entropy(probs)   # Shannon entropy in nats
normalized_entropy = entropy / np.log(len(probs))
```

We replicate this here and verify it matches the stored analysis values.

In [ ]:
def compute_current_entropy(responses):
    """Replicates analysis_utils.get_human_probs + scipy.stats.entropy.
    
    Returns entropy in nats (scipy default), normalized entropy, and probs.
    """
    unique, counts = np.unique(responses, return_counts=True)
    probs = counts / counts.sum()
    entropy_nats = stats.entropy(probs)          # nats (natural log)
    normalized = entropy_nats / np.log(len(probs))
    return entropy_nats, normalized, probs, unique


results = []
for word_idx, group in df_sub.groupby('word_index'):
    responses = group['response'].dropna().values
    responses = responses[responses != '']
    h, h_norm, probs, unique = compute_current_entropy(responses)
    results.append({
        'word_index': word_idx,
        'n_responses': len(responses),
        'n_unique': len(unique),
        'entropy_scipy': h,
        'norm_entropy_scipy': h_norm,
    })

df_compare = pd.DataFrame(results)
df_compare.head()

In [ ]:
# Cross-check against stored analysis values
df_ref = df_analyzed[
    (df_analyzed['task'] == TASK) & (df_analyzed['modality'] == MODALITY)
][['word_index', 'entropy', 'normalized_entropy']]
df_ref = df_ref[df_ref['word_index'].isin(sample_words)]

df_compare = df_compare.merge(df_ref, on='word_index')

max_diff = (df_compare['entropy_scipy'] - df_compare['entropy']).abs().max()
print(f'Max |entropy_scipy - stored entropy|: {max_diff:.2e}')
print('(Should be < 1e-10 if replication is exact)')

df_compare[['word_index', 'n_responses', 'n_unique', 'entropy_scipy', 'entropy']].round(6)

## 3. Verify infomeasure discrete matches scipy

**Key API difference:**
- `scipy.stats.entropy(probs)` — takes a probability vector, outputs nats
- `infomeasure.entropy(samples, approach='discrete', base='e')` — takes raw samples (strings OK), outputs nats

Both are MLE plug-in estimators and should agree up to floating-point precision.

In [ ]:
for word_idx, group in df_sub.groupby('word_index'):
    responses = group['response'].dropna().values
    responses = responses[responses != '']
    h_im = im.entropy(responses, approach='discrete', base='e')
    df_compare.loc[df_compare['word_index'] == word_idx, 'entropy_im_discrete'] = h_im

max_diff_im = (df_compare['entropy_scipy'] - df_compare['entropy_im_discrete']).abs().max()
print(f'Max |entropy_scipy - infomeasure discrete|: {max_diff_im:.2e}')
print('(Should be ~0 — both are MLE plug-in estimators)')

df_compare[['word_index', 'n_unique', 'entropy_scipy', 'entropy_im_discrete']].round(8)

## 4. Bias-corrected entropy estimators

For N≈50 discrete samples, the MLE plug-in estimator systematically underestimates entropy. We compare four corrected methods:

| Method | Key property |
|--------|-------------|
| **Miller-Madow** | Simple additive correction: `H_MLE + (K-1)/(2N)` |
| **Shrinkage (James-Stein)** | Regularization toward uniform; MSE-optimal for independent data |
| **NSB** | Bayesian with numerical integration; robust for correlated/sequential data |
| **Chao-Shen** | Coverage-based; accounts for unobserved vocabulary items |

For this data (word responses from ~50 participants), **NSB** and **Shrinkage** are the primary candidates.

In [ ]:
approaches = {
    'miller_madow': 'Miller-Madow',
    'shrink':       'Shrinkage',
    'nsb':          'NSB',
    'chao_shen':    'Chao-Shen',
}

for approach, label in approaches.items():
    col = f'entropy_{approach}'
    for word_idx, group in df_sub.groupby('word_index'):
        responses = group['response'].dropna().values
        responses = responses[responses != '']
        try:
            h = im.entropy(responses, approach=approach, base='e')
        except Exception as e:
            print(f'  [{approach}] word_index={word_idx}: {e}')
            h = np.nan
        df_compare.loc[df_compare['word_index'] == word_idx, col] = h
    print(f'{label}: done')

print('\nAll estimators computed.')

## 5. Summary table

In [ ]:
entropy_cols = [
    'entropy_scipy',
    'entropy_miller_madow',
    'entropy_shrink',
    'entropy_nsb',
    'entropy_chao_shen',
]
labels = {
    'entropy_scipy':        'MLE (scipy/current)',
    'entropy_miller_madow': 'Miller-Madow',
    'entropy_shrink':       'Shrinkage',
    'entropy_nsb':          'NSB',
    'entropy_chao_shen':    'Chao-Shen',
}

summary = pd.DataFrame({
    'Method': [labels[c] for c in entropy_cols],
    'Mean (nats)': [df_compare[c].mean() for c in entropy_cols],
    'Std (nats)':  [df_compare[c].std()  for c in entropy_cols],
    'Δ vs MLE (mean)': [
        (df_compare[c] - df_compare['entropy_scipy']).mean()
        for c in entropy_cols
    ],
})

summary = summary.set_index('Method').round(4)
print(summary.to_string())

## 6. Visualizations

In [ ]:
# Line plot: entropy per word_index for each method
fig, ax = plt.subplots(figsize=(12, 4))

colors = {
    'entropy_scipy':        ('#555555', '-',  'MLE (current)'),
    'entropy_miller_madow': ('#4C72B0', '--', 'Miller-Madow'),
    'entropy_shrink':       ('#DD8452', '-',  'Shrinkage'),
    'entropy_nsb':          ('#55A868', '-',  'NSB'),
    'entropy_chao_shen':    ('#C44E52', '--', 'Chao-Shen'),
}

for col, (color, ls, label) in colors.items():
    ax.plot(df_compare['word_index'], df_compare[col],
            color=color, linestyle=ls, marker='o', markersize=4, label=label, alpha=0.85)

ax.set_xlabel('Word index')
ax.set_ylabel('Entropy (nats)')
ax.set_title(f'Entropy by method — task={TASK}, modality={MODALITY}')
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left')
sns.despine(ax=ax)
plt.tight_layout()
plt.show()

In [ ]:
# Scatter plots: MLE vs each corrected estimator
corrected_cols = [
    ('entropy_miller_madow', 'Miller-Madow'),
    ('entropy_shrink',       'Shrinkage'),
    ('entropy_nsb',          'NSB'),
    ('entropy_chao_shen',    'Chao-Shen'),
]

fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))

mle = df_compare['entropy_scipy']
lims = (mle.min() - 0.05, df_compare[entropy_cols].max().max() + 0.05)

for ax, (col, label) in zip(axes, corrected_cols):
    corr = df_compare[col]
    ax.scatter(mle, corr, alpha=0.8, edgecolors='w', linewidths=0.5)
    ax.plot(lims, lims, 'k--', lw=1, alpha=0.5, label='y=x')
    ax.set_xlim(lims)
    ax.set_ylim(lims)
    ax.set_xlabel('MLE entropy (nats)')
    ax.set_ylabel(f'{label} (nats)')
    ax.set_title(label)
    ax.set_aspect('equal')
    mean_diff = (corr - mle).mean()
    ax.text(0.05, 0.92, f'Δ={mean_diff:+.3f} nats', transform=ax.transAxes,
            fontsize=9, color='darkred')
    sns.despine(ax=ax)

plt.suptitle(f'MLE vs corrected estimators — task={TASK}, modality={MODALITY}', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Bias correction magnitude vs sample properties
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, (col, label) in zip(axes, [('entropy_shrink', 'Shrinkage'), ('entropy_nsb', 'NSB')]):
    diff = df_compare[col] - df_compare['entropy_scipy']
    ax.scatter(df_compare['n_unique'], diff, alpha=0.8)
    ax.axhline(0, color='k', lw=0.8, linestyle='--')
    ax.set_xlabel('Number of unique responses')
    ax.set_ylabel('Δ entropy (corrected − MLE, nats)')
    ax.set_title(f'Bias correction: {label}')
    sns.despine(ax=ax)

plt.suptitle('Bias correction magnitude vs vocabulary size', y=1.02)
plt.tight_layout()
plt.show()

## 7. Discussion

**Sanity check (Sections 3–4):**
- The scipy replication should match the stored analysis values (max diff < 1e-10), confirming the pipeline is correctly reproduced
- The `infomeasure` discrete estimator should agree with scipy up to floating-point precision (~1e-15), confirming equivalent MLE formulations

**Bias correction (Section 5):**
- All corrected estimators should return values ≥ MLE (bias correction is always upward for entropy)
- **Miller-Madow** provides the smallest, most conservative correction
- **Shrinkage** and **NSB** provide larger corrections and are the primary candidates
- **Chao-Shen** may correct aggressively when there are many singleton responses (words seen only once)

**Choosing between NSB and Shrinkage:**
- If responses across participants are largely independent: **Shrinkage** (James-Stein) is MSE-optimal
- If responses are correlated (e.g., neighboring participants echo each other): **NSB** is more robust
- For word prediction tasks with independent participants, **Shrinkage** is likely appropriate

**Next steps before applying to full pipeline:**
- Verify results hold across all tasks and modalities
- Assess whether bias correction changes any substantive conclusions (entropy group assignments, correlations)
- NSB is computationally heavier — benchmark on full dataset before committing

In [ ]:
# Full comparison table
display_cols = ['word_index', 'n_responses', 'n_unique',
                'entropy_scipy', 'entropy_miller_madow',
                'entropy_shrink', 'entropy_nsb', 'entropy_chao_shen']
df_compare[display_cols].round(4)